In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# 작업 모니터링 또는 취소

[워크로드 페이지](https://quantum.cloud.ibm.com/workloads)에서 워크로드 목록을 확인할 수 있습니다.

## 작업 상태 보기
[워크로드 테이블](https://quantum.cloud.ibm.com/workloads)로 이동하여 상태 열에서 작업이 완료되었는지 또는 실패했는지 확인합니다.

## 남은 사용량 보기
[인스턴스 테이블](https://quantum.cloud.ibm.com/instances)로 이동하여 남은 사용량을 확인하려는 플랜과 연관된 탭을 선택합니다. 플랜에서 사용된 총 시간과 남은 총 시간이 표시됩니다.

## 제출된 작업 및 워크로드 수에 대한 지표 보기
[분석 페이지](https://quantum.cloud.ibm.com/analytics)로 이동하여 제출된 총 작업 수와 함께 배치 워크로드 및 세션 워크로드의 수를 확인합니다. 분석 페이지는 소유하거나 관리하는 계정에 대해서만 볼 수 있습니다.

## 작업 모니터링
작업 인스턴스를 사용하여 작업 상태를 확인하거나 적절한 명령을 호출하여 결과를 검색합니다.

|                               |                                                                                                                                                                                                              |
| ----------------------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| job.result()                  | 작업이 완료된 직후 작업 결과를 검토합니다. 작업 결과는 작업이 완료된 후에 사용할 수 있습니다. 따라서 job.result()는 작업이 완료될 때까지 블로킹 호출입니다.                               |
| job.job\_id()                 | 해당 작업을 고유하게 식별하는 ID를 반환합니다. 나중에 작업 결과를 검색하려면 작업 ID가 필요합니다. 따라서 나중에 검색할 작업의 ID를 저장해 두는 것이 좋습니다. |
| job.status()                  | 작업 상태를 확인합니다.                                                                                                                                                                                        |
| job = service.job(\<job\_id>) | 이전에 제출한 작업을 검색합니다. 이 호출에는 작업 ID가 필요합니다.                                                                                                                                      |

<span id="retrieve-later"></span>
## 나중에 작업 결과 검색
`service.job(\<job\_id>)`를 호출하여 이전에 제출한 작업을 검색합니다. 작업 ID가 없거나 여러 작업을 한 번에 검색하려는 경우(퇴역한 QPU(양자 처리 장치)의 작업 포함), 선택적 필터를 사용하여 `service.jobs()`를 호출합니다. [QiskitRuntimeService.jobs](../api/qiskit-ibm-runtime/qiskit-runtime-service#jobs)를 참조하세요.

> **Note:** `service.jobs()`는 더 이상 사용되지 않는 `qiskit-ibm-provider` 패키지에서 실행된 작업도 반환합니다. 이전(역시 더 이상 사용되지 않는) `qiskit-ibmq-provider` 패키지로 제출된 작업은 더 이상 사용할 수 없습니다.

### 예시
이 예시는 `my_backend`에서 실행된 가장 최근 런타임 작업 10개를 반환합니다.

In [1]:
# This cell is hidden from users
from qiskit import QuantumCircuit
from qiskit.circuit import Parameter
from qiskit.transpiler import generate_preset_pass_manager

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
import numpy as np


my_backend = "ibm_torino"
service = QiskitRuntimeService()
# backend = service.backend(my_backend)
backend = service.least_busy()

# Define two circuits, each with one parameter with two parameters.
circuit = QuantumCircuit(2)
circuit.h(0)
circuit.cx(0, 1)
circuit.ry(Parameter("a"), 0)
circuit.cx(0, 1)
circuit.h(0)
circuit.measure_all()


pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
transpiled_circuit = pm.run(circuit)

params = np.random.uniform(size=(2, 3)).T

sampler_pub = (transpiled_circuit, params)

# Instantiate the new estimator object, then run the transpiled circuit
# using the set of parameters and observables.
sampler = SamplerV2(mode=backend)
job = sampler.run([sampler_pub], shots=4)
print(job.job_id())

d305ck0ocacs73ajagvg


In [2]:
result = job.result()


spans = job.result().metadata["execution"]["execution_spans"]
print(spans)

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])


In [3]:
params = np.random.uniform(size=(2, 3))
params

array([[0.2260416 , 0.8747859 , 0.44361995],
       [0.94700856, 0.96826017, 0.98426562]])

In [4]:
mask = spans[0].mask(0)
mask

array([[[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]],

       [[ True,  True,  True,  True],
        [ True,  True,  True,  True]]])

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

# Initialize the account first.
service = QiskitRuntimeService()
# Use `limit` to retrieve a specific number of jobs. The default `limit` is 10.
service.jobs(backend_name=my_backend)

## Cancel a job

You can cancel a job from the IBM Quantum Platform dashboard either on the Workloads page or the details page for a specific workload. On the Workloads page, click the overflow menu at the end of the row for that workload, and select Cancel. If you are on the details page for a specific workload, use the Actions dropdown at the top of the page, and select Cancel.

In Qiskit, use `job.cancel()` to cancel a job.

<span id="execution-spans"></span>
## View Sampler execution spans

The results of [`SamplerV2`](/docs/api/qiskit-ibm-runtime/sampler-v2) jobs executed in Qiskit Runtime contain execution timing information in their metadata.
This timing information can be used to place upper and lower timestamp bounds on when particular shots were executed on the QPU.
Shots are grouped into [`ExecutionSpan`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span) objects, each of which indicates a start time, a stop time, and a specification of which shots were collected in the span.

An execution span specifies which data was executed during its window by providing an [`ExecutionSpan.mask`](/docs/api/qiskit-ibm-runtime/execution-span-execution-span#mask) method. This method, given any [Primitive Unified Block (PUB)](/docs/guides/primitive-input-output) index, returns a boolean mask that is `True` for all shots executed during its window. PUBs are indexed by the order in which they were given to the Sampler run call. If, for example, a PUB has shape `(2, 3)` and was run with four shots, then the mask's shape is `(2, 3, 4)`. See the [execution_span](/docs/api/qiskit-ibm-runtime/execution-span) API page for full details.

Example:

To view execution span information, review the metadata of the result returned by `SamplerV2`, which comes in the form of an `ExecutionSpans` object. This object is a list-like container containing instances of subclasses of `ExecutionSpan`, such as `SliceSpan`:

In [6]:
from qiskit.primitives import BitArray

# Get the mask of the 1st PUB for the 0th span.
mask = spans[0].mask(0)

# Decide whether the 0th shot of parameter set (1, 2) occurred in this span.
in_this_span = mask[2, 1, 0]

# Create a new bit array containing only the PUB-1 data collected during this span.
bits = result[0].data.meas
filtered_data = BitArray(bits.array[mask], bits.num_bits)

Execution spans can be filtered to include information pertaining to specific PUBs, selected by their indices:

In [7]:
# take the subset of spans that reference data in PUBs 0 or 2
spans.filter_by_pub([0, 2])

ExecutionSpans([DoubleSliceSpan(<start='2025-09-09 16:31:16', stop='2025-09-09 16:31:16', size=24>)])

View global information about the collection of execution spans:

In [8]:
print("Number of execution spans:", len(spans))
print("  Start of the first span:", spans.start)
print("     End of the last span:", spans.stop)
print("       Total duration (s):", spans.duration)

Number of execution spans: 1
  Start of the first span: 2025-09-09 16:31:16.320568
     End of the last span: 2025-09-09 16:31:16.865858
       Total duration (s): 0.54529


Extract and inspect a particular span:

In [9]:
spans.sort()
print(" Start of first span:", spans[0].start)
print("   End of first span:", spans[0].stop)
print("#shots in first span:", spans[0].size)

 Start of first span: 2025-09-09 16:31:16.320568
   End of first span: 2025-09-09 16:31:16.865858
#shots in first span: 24


## 작업 취소
IBM Quantum Platform 대시보드의 워크로드 페이지 또는 특정 워크로드의 세부 정보 페이지에서 작업을 취소할 수 있습니다. 워크로드 페이지에서 해당 워크로드 행 끝의 오버플로 메뉴를 클릭하고 취소를 선택합니다. 특정 워크로드의 세부 정보 페이지에 있는 경우 페이지 상단의 작업 드롭다운을 사용하여 취소를 선택합니다.

Qiskit에서는 `job.cancel()`을 사용하여 작업을 취소합니다.

<span id="execution-spans"></span>
## Sampler 실행 스팬 보기
Qiskit Runtime에서 실행된 [`SamplerV2`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2) 작업의 결과에는 메타데이터에 실행 타이밍 정보가 포함되어 있습니다.
이 타이밍 정보는 특정 샷이 QPU에서 실행된 시간의 상한 및 하한 타임스탬프를 설정하는 데 사용할 수 있습니다.
샷은 [`ExecutionSpan`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span) 객체로 그룹화되며, 각 객체는 시작 시간, 종료 시간, 그리고 해당 스팬에서 수집된 샷의 사양을 나타냅니다.

실행 스팬은 [`ExecutionSpan.mask`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span-execution-span#mask) 메서드를 제공하여 해당 윈도우 동안 실행된 데이터를 지정합니다. 이 메서드는 [Primitive Unified Block (PUB)](/guides/primitive-input-output) 인덱스를 인수로 받아 해당 윈도우 동안 실행된 모든 샷에 대해 `True`인 불리언 마스크를 반환합니다. PUB는 Sampler 실행 호출에 제공된 순서로 인덱싱됩니다. 예를 들어, PUB의 형태가 `(2, 3)`이고 4개의 샷으로 실행된 경우 마스크의 형태는 `(2, 3, 4)`입니다. 전체 세부 사항은 [execution_span](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/execution-span) API 페이지를 참조하세요.

예시:
실행 스팬 정보를 보려면 `SamplerV2`가 반환한 결과의 메타데이터를 검토합니다. 이는 `ExecutionSpans` 객체 형태로 제공됩니다. 이 객체는 `SliceSpan`과 같은 `ExecutionSpan`의 서브클래스 인스턴스를 포함하는 리스트형 컨테이너입니다.